# 单站客流的滚动一步预测

在本 Notebook 所在目录运行，确认旁边有 LSTM.csv。学习目标是用已观测的五点历史预测下一点，并在原始人数尺度与上一观测基线比较。从上到下执行；下列编号与网页中的原代码单元对应。

## 数据 · 原单元 5–12

读入 data 列后，检查有效观测数和字段。文件给出 900 条有序数值。能从这一列确认每条观测的日期和频率吗？

<details><summary>核对思路</summary>

文件没有独立时间戳。按观测下标报告切分；缩放的极值只取自训练窗口覆盖的观测。

</details>

In [ ]:
%matplotlib inline
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)

In [ ]:
data_csv = pd.read_csv("./LSTM.csv")
data_csv.head()

In [ ]:
plt.plot(data_csv[0:200])

In [ ]:
raw_flow = data_csv['data'].dropna().to_numpy(dtype=np.float32)
step = 5
train_size = int((len(raw_flow) - step) * 0.7)
if train_size < 1 or train_size >= len(raw_flow) - step:
    raise ValueError('需要足够的数据以构造非空的训练集和测试集')
training_values = raw_flow[:train_size + step]
min_value = training_values.min()
max_value = training_values.max()
scale = max_value - min_value
if scale == 0:
    scale = np.float32(1.0)
dataset = (raw_flow - min_value) / scale

## 建窗 · 原单元 14–22

画出前两个五点窗口，标出各自的目标。训练有 626 个窗口，第一个测试目标的零基下标是多少？先写下答案，再运行形状检查。

<details><summary>核对思路</summary>

首个测试窗口读 626–630，标签是 631。测试张量 (269, 1, 5) 表示 269 个顺次到达的窗口、1 条序列、每步 5 个历史特征。

</details>

In [ ]:
def create_dataset(series, step):
    windows = np.lib.stride_tricks.sliding_window_view(series, step + 1)
    return windows[:, :-1].copy(), windows[:, -1].copy()

In [ ]:
data_X, data_Y = create_dataset(dataset, step)

In [ ]:
train_X, test_X = data_X[:train_size], data_X[train_size:]
train_Y, test_Y = data_Y[:train_size], data_Y[train_size:]

In [ ]:
train_X.shape

In [ ]:
# 保留 (时间步, 批次, 特征) 的数据表示，在模型中转为批次优先。
train1 = tf.convert_to_tensor(train_X.reshape(-1, 1, step))
train2 = tf.convert_to_tensor(train_Y.reshape(-1, 1, 1))
test1 = tf.convert_to_tensor(test_X.reshape(-1, 1, step))
test2 = tf.convert_to_tensor(test_Y.reshape(-1, 1, 1))

In [ ]:
train1.shape, train2.shape, test1.shape, test2.shape

## 模型和训练 · 原单元 24–28

追踪一次前向传播。写出模型收到的序列长度、每步特征数和输出宽度。完整训练为 200 轮。

<details><summary>核对思路</summary>

外部输入遵循 (时间步, 批次, 特征)，在模型内转为 Keras 的批次优先顺序，输出再转回。

</details>

In [ ]:
class lstm_linear(tf.keras.Model):
    def __init__(self, input_size, hidden_size, output_size=1, num_layers=2):
        super().__init__()
        self.lstm = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(None, input_size)),
            *[tf.keras.layers.LSTM(hidden_size, return_sequences=True) for _ in range(num_layers)],
        ])
        self.linear = tf.keras.layers.Dense(output_size)

    def call(self, x, training=False):
        batch_first = tf.transpose(x, (1, 0, 2))
        hidden = self.lstm(batch_first, training=training)
        return tf.transpose(self.linear(hidden), (1, 0, 2))

In [ ]:
net = lstm_linear(step, 45)

criterion = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-2, epsilon=1e-8)

In [ ]:
@tf.function
def train_step(inputs, targets):
    with tf.GradientTape() as tape:
        loss = criterion(targets, net(inputs, training=True))
    gradients = tape.gradient(loss, net.trainable_variables)
    optimizer.apply_gradients(zip(gradients, net.trainable_variables))
    return loss


for e in range(200):
    loss = train_step(train1, train2)
    if (e + 1) % 5 == 0:
        print(f'Epoch: {e + 1}, Loss: {float(loss):.5f}')

## 测试 · 原单元 30–36

先判断：若使用固定起点向后预测，第二个窗口的最后一点应取什么？再运行本例的滚动一步测试，核对预测、标签与上一观测长度。

<details><summary>核对思路</summary>

固定起点递归时使用第一步预测。本例使用此时已经观测到的真实值，269 个预测对应相同数量的标签。上一观测 MSE 约为 476672.15 人²。对 MSE 开方得到单位为人的 RMSE。

</details>

In [ ]:
# 测试时调用模型并传入 training=False。

In [ ]:
test_predictions = net(test1, training=False)
normalized_test_mse = float(criterion(test2, test_predictions))
print(f'Normalized test MSE: {normalized_test_mse:.6f}')

In [ ]:
pred_test = test_predictions.numpy().reshape(-1).astype(np.float64) * scale + min_value
true_test = raw_flow[train_size + step:].astype(np.float64)
last_observed = raw_flow[train_size + step - 1:-1].astype(np.float64)

In [ ]:
assert pred_test.shape == true_test.shape == last_observed.shape
print(f'Train windows: {len(train1)}, test windows: {len(test1)}')

In [ ]:
mse = np.mean((pred_test - true_test) ** 2)
rmse = np.sqrt(mse)
persistence_mse = np.mean((last_observed - true_test) ** 2)
print(f'Test MSE (people²): {mse:.6f}')
print(f'Test RMSE (people): {rmse:.6f}')
print(f'Last observation MSE (people²): {persistence_mse:.6f}')

In [ ]:
# 画出实际结果和预测的结果
plt.plot(true_test,  label='real')
plt.plot(pred_test,label='prediction')
plt.legend(loc='best')